# 评分新增基线并更新结果

读取正式生成完成的新增方法，用原 A1、A2、A3 指标评分。每个结果通过配额、区域轴与数值域核验后，更新对应主实验表。使用已保存的真实训练参照 ANOVA 选区和填补中位数重建 PCA。旧方法沿用已有结果。Liu2023 全量新增条目进入四幅主实验图；每次发布都从 CSV 重新生成 README。

In [ ]:
from pathlib import Path
import gc, json, os, shutil, sys
import anndata as ad
import nbformat
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from threadpoolctl import threadpool_limits
from data.baseline import dense32
from data.evaluation import dense_block, imputed_features, read_labels, region_pairs
from data.paths import OUTPUTS, PROJECT, check_output
from evaluate.A1.metrics import a1_metrics
from evaluate.A1.plotting import load_plot_regions, plot_a1
from evaluate.A2.metrics import paired_pearson
from evaluate.A3.metrics import milisi

METHODS=['simscDNAm','FittedMarginal','simATAC','simCAS','PeakVI','DiTSim']
NEW=METHODS[4:]
SUBSETS={'liu2023':['GSE213221','GSE215213','GSE215342'],
         'liu2021':['GSE130553','GSE131354','GSE131360'],
         'tian2023':['GSE167119','GSE167577','GSE167978']}
CASES=['liu2023/full']+[f'{study}/{gse}' for study,accessions in SUBSETS.items() for gse in accessions]
COLUMNS=['A1 1-WD↓','A1 MSE↓','A1 Pearson↑','A2 Pearson↑','A3 miLISI↑']
RUN=OUTPUTS/'internal/new_baselines'
FIGURE_PREFIX='/files/data/xb/outputs/simscDNAm/figures/'

def read_csv(path):
    return pd.read_csv(path,float_precision='round_trip')

def write_csv(frame,path):
    path=check_output(path)
    temporary=path.with_suffix('.csv.partial')
    frame.to_csv(temporary,index=False,encoding='utf-8-sig')
    temporary.replace(path)

def notebook_function(name, function):
    scope={}
    for cell in nbformat.read(PROJECT/'notebooks'/name,as_version=4).cells[:-1]:
        if cell.cell_type=='code':exec(cell.source,scope)
    return scope[function]

def score(case, method):
    assert case in CASES and method in NEW
    destination=RUN/case/method
    destination.mkdir(parents=True,exist_ok=True)
    if (destination/'scores.csv').exists():
        return
    prepared=OUTPUTS/'preprocessing'/case
    original=OUTPUTS/'internal/evaluation'/case
    candidate_path=OUTPUTS/'baseline'/case/method/'data.h5ad'
    with threadpool_limits(limits=8):
        real=ad.read_h5ad(prepared/'HeldoutReal.h5ad')
        candidate=ad.read_h5ad(candidate_path)
        assert candidate.shape==real.shape and candidate.var_names.equals(real.var_names)
        assert candidate.obs_names.is_unique
        pd.testing.assert_series_equal(candidate.obs['Subclass'].astype(str).value_counts().sort_index(),
            real.obs['Subclass'].astype(str).value_counts().sort_index(),check_names=False)
        for start in range(0,candidate.n_obs,32):
            block=dense32(candidate.X[start:start+32])
            assert np.isfinite(block).all() and block.min()>=0 and block.max()<=1
        print('核验并计算 A1',case,method,flush=True)
        natural=real.n_vars-1000
        real_rows=np.arange(real.n_obs)
        rows=np.arange(candidate.n_obs)
        a1,details=a1_metrics(real,candidate,real_rows,rows,natural)
        write_csv(details,destination/'A1_type_details.csv')
        pairs=region_pairs(real,pair_path=prepared/'region_pairs.csv')
        a2=pd.concat([pairs,paired_pearson(dense_block(candidate,rows,pairs['原region索引'].to_numpy()),
             dense_block(candidate,rows,pairs['复制region索引'].to_numpy()))],axis=1)
        a2.insert(0,'方法',method)
        write_csv(a2,destination/'A2_duplicate_correlations.csv')
        print('计算 A3',case,method,flush=True)
        cached=RUN/case/'pca.npz'
        if not cached.exists():
            train=ad.read_h5ad(prepared/'TrainingReal.h5ad')
            assert not len(train.obs_names.intersection(real.obs_names))
            frozen=read_csv(original/'A3_selected_regions.csv')
            selected=train.var_names.get_indexer(frozen['原region'])
            assert len(selected)==50000 and selected[-1]<natural and selected[0]>=0 and np.all(np.diff(selected)>0)
            fill=np.full(natural,np.nan)
            fill[selected]=frozen['填补中位数'].to_numpy()
            pca=PCA(n_components=min(50,train.n_obs-1),svd_solver='randomized',random_state=42)
            pca.fit(imputed_features(train,np.arange(train.n_obs),selected,fill))
            del train
            np.savez(cached, selected=selected, fill=fill, components=pca.components_, mean=pca.mean_,
                     explained_variance_ratio=pca.explained_variance_ratio_)
        with np.load(cached) as cache:
            selected, fill = cache['selected'],cache['fill']
            pca=PCA(n_components=len(cache['components']),svd_solver='randomized',random_state=42)
            pca.components_=cache['components']
            pca.mean_=cache['mean']
            pca.explained_variance_ratio_=cache['explained_variance_ratio']
            pca.n_features_in_=len(selected)
        real_rep=pca.transform(imputed_features(real,real_rows,selected,fill))
        candidate_rep=pca.transform(imputed_features(candidate,rows,selected,fill))
        a3,cells,types=milisi(real_rep,read_labels(real,real_rows),candidate_rep,read_labels(candidate,rows),30)
        write_csv(cells,destination/'A3_cells.csv')
        write_csv(types,destination/'A3_type_details.csv')
        if case=='liu2023/full':
            import umap
            xy=umap.UMAP(n_neighbors=30,random_state=2022,transform_seed=2022,n_jobs=1).fit_transform(np.vstack([real_rep,candidate_rep]))
            coordinates=pd.DataFrame({'方法':method,'来源':np.repeat(['真实','合成'],[real.n_obs,candidate.n_obs]),
                '细胞ID':np.r_[real.obs_names,candidate.obs_names],'UMAP1':xy[:,0],'UMAP2':xy[:,1]})
            write_csv(coordinates,destination/'A3_joint_coordinates.csv')
            (destination/'figures').mkdir(exist_ok=True)
            regions=load_plot_regions(real,natural,PROJECT/'evaluate/A1/marker_regions_liu2023.csv','mm10')
            plot_a1({'真实数据':real,method:candidate},{'真实数据':real_rows,method:rows},[method],regions,destination)
        row={'子数据集':case.split('/')[1],'方法':method,COLUMNS[0]:a1['加权 1-WD'],COLUMNS[1]:a1['加权 MSE'],
             COLUMNS[2]:a1['加权 Pearson'],COLUMNS[3]:a2['Pearson'].mean(),COLUMNS[4]:a3}
        write_csv(pd.DataFrame([row]),destination/'scores.csv')
    print('评分完成',case,method,flush=True)

def publish():
    all_tables={}
    for case in CASES:
        original=OUTPUTS/'internal/evaluation'/case
        old=read_csv(original/'scores.csv')
        old=old.loc[old['方法'].isin(METHODS[:4])].reset_index(drop=True)
        assert old['方法'].tolist()==METHODS[:4]
        scores=[RUN/case/method/'scores.csv' for method in NEW]
        table=pd.concat([old,*[read_csv(path) for path in scores if path.is_file()]],ignore_index=True)
        pd.testing.assert_frame_equal(table.iloc[:4].reset_index(drop=True),old,check_exact=True)
        all_tables[case]=table
        study, accession=case.split('/')
        title=f"{study} {'全数据集' if accession=='full' else accession}"
        write_csv(table.drop(columns='子数据集'),OUTPUTS/'逐数据集表'/f'{title}.csv')
    full=all_tables['liu2023/full']
    previous=read_csv(OUTPUTS/'tables/liu2023全数据集.csv')['方法'].tolist()
    write_csv(full.drop(columns='子数据集'),OUTPUTS/'tables/liu2023全数据集.csv')
    for study, accessions in SUBSETS.items():
        write_csv(pd.concat([all_tables[f'{study}/{gse}'] for gse in accessions],ignore_index=True),
                  OUTPUTS/'tables'/f'{study}三个子数据集.csv')
    if full['方法'].tolist()!=previous:
        original=OUTPUTS/'internal/evaluation/liu2023/full'
        for name in ['A1_plot_histograms.csv','A1_type_histograms.csv','A2_duplicate_correlations.csv','A3_joint_coordinates.csv']:
            retained=read_csv(original/name)
            retained=retained.loc[~retained['方法'].isin(NEW)]
            added=[read_csv(RUN/'liu2023/full'/method/name) for method in full['方法'] if method in NEW]
            added=[part.loc[part['方法'].isin(NEW)] for part in added]
            write_csv(pd.concat([retained,*added],ignore_index=True),original/name)
        notebook_function('比较图重绘.ipynb','redraw_comparison')(OUTPUTS)
    finished=notebook_function('更新结果README.ipynb','update_results_readme')(OUTPUTS,FIGURE_PREFIX)
    print('README 和十一张表已刷新；新增已评分组合',finished,f'/{len(CASES) * len(NEW)}',flush=True)
    report_path=OUTPUTS/'最终交付核验.json'
    report=json.loads(report_path.read_text())
    report.update({'主实验表':10,'消融表':1,'全量图':4,'消融图':2,'图像数':6,
        '已评分新增组合':finished,'主实验方法行':40+finished,'五项指标总值数':(44+finished)*5,
        '状态':'通过' if finished == len(CASES) * len(NEW) else '新增基线实验中'})
    report_path.write_text(json.dumps(report,ensure_ascii=False,indent=2))
    return finished


In [ ]:
for case in CASES:
    for method in NEW:
        record=RUN/f"{case.replace('/','__')}__{method}.json"
        if record.exists() and json.loads(record.read_text()).get('状态')=='完成':
            score(case, method)
publish()